# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
# Same cleaning and feature engineering as the previous lab, so the comparison is fair

# 1) Remove rows with missing values
spaceship = spaceship.dropna()

# 2) Keep only the deck letter of the cabin ("B/0/P" -> "B")
spaceship["Cabin"] = spaceship["Cabin"].str[0]

# 3) Feature selection: remove columns that can't help predict (unique for everyone)
spaceship = spaceship.drop(columns=["PassengerId", "Name"])

# 4) Separate features (X) and target (y)
X = spaceship.drop(columns=["Transported"])
y = spaceship["Transported"]

# 5) Turn text / True-False columns into 0/1 columns (dummies)
X = pd.get_dummies(X, columns=["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"],
                   drop_first=True, dtype=int)
X.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,1,0,0,1,0,0,0,0,0,0,0,1,0
1,24.0,109.0,9.0,25.0,549.0,44.0,0,0,0,0,0,0,0,1,0,0,0,1,0
2,58.0,43.0,3576.0,0.0,6715.0,49.0,1,0,0,0,0,0,0,0,0,0,0,1,1
3,33.0,0.0,1283.0,371.0,3329.0,193.0,1,0,0,0,0,0,0,0,0,0,0,1,0
4,16.0,303.0,70.0,151.0,565.0,2.0,0,0,0,0,0,0,0,1,0,0,0,1,0


**Perform Train Test Split**

In [4]:
# 80% of the data to train the models, 20% to test them
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling: put every column between 0 and 1
# We fit the scaler on TRAIN only (to avoid data leakage), then apply it to train and test
from sklearn.preprocessing import MinMaxScaler

normalizer = MinMaxScaler()
normalizer.fit(X_train)

X_train_norm = normalizer.transform(X_train)
X_test_norm = normalizer.transform(X_test)

print("Train:", X_train_norm.shape)
print("Test: ", X_test_norm.shape)

Train: (5284, 19)
Test:  (1322, 19)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [5]:
# Bagging and Pasting = train many decision trees, each on a random sample of the data,
# then the trees VOTE for the final answer (majority wins)
# - Bagging: samples are taken WITH replacement (bootstrap=True) -> a row can be picked several times
# - Pasting: samples are taken WITHOUT replacement (bootstrap=False) -> each row at most once per tree
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Bagging
bagging = BaggingClassifier(DecisionTreeClassifier(max_depth=10),
                            n_estimators=100,     # 100 trees
                            max_samples=1000,     # each tree sees 1000 random rows
                            bootstrap=True,       # WITH replacement = bagging
                            random_state=42)
bagging.fit(X_train_norm, y_train)
pred_bagging = bagging.predict(X_test_norm)
print("Bagging accuracy:", round(accuracy_score(y_test, pred_bagging), 3))

# Pasting
pasting = BaggingClassifier(DecisionTreeClassifier(max_depth=10),
                            n_estimators=100,
                            max_samples=1000,
                            bootstrap=False,      # WITHOUT replacement = pasting
                            random_state=42)
pasting.fit(X_train_norm, y_train)
pred_pasting = pasting.predict(X_test_norm)
print("Pasting accuracy:", round(accuracy_score(y_test, pred_pasting), 3))

Bagging accuracy: 0.809
Pasting accuracy: 0.804


- Random Forests

In [6]:
# Random Forest = bagging of decision trees + extra randomness:
# at each split, each tree only looks at a random subset of the columns
# -> the trees are more different from each other, so their vote is stronger
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100,   # 100 trees
                                max_depth=10,       # each tree has at most 10 levels
                                random_state=42)
forest.fit(X_train_norm, y_train)
pred_forest = forest.predict(X_test_norm)
print("Random Forest accuracy:", round(accuracy_score(y_test, pred_forest), 3))

Random Forest accuracy: 0.815


- Gradient Boosting

In [7]:
# Gradient Boosting = trees are built ONE AFTER THE OTHER
# each new tree tries to correct the errors (the "residuals") left by the previous trees
# learning_rate = how much each new tree is allowed to correct (small = slower but safer)
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100,
                                max_depth=3,
                                learning_rate=0.1,
                                random_state=42)
gb.fit(X_train_norm, y_train)
pred_gb = gb.predict(X_test_norm)
print("Gradient Boosting accuracy:", round(accuracy_score(y_test, pred_gb), 3))

Gradient Boosting accuracy: 0.807


- Adaptive Boosting

In [8]:
# Adaptive Boosting (AdaBoost) = trees are also built one after the other,
# but after each tree, the passengers it got WRONG get a bigger weight
# -> the next tree focuses more on the difficult passengers
# At the end, better trees get a bigger say in the final vote
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=3),
                         n_estimators=100,
                         learning_rate=0.5,
                         random_state=42)
ada.fit(X_train_norm, y_train)
pred_ada = ada.predict(X_test_norm)
print("AdaBoost accuracy:", round(accuracy_score(y_test, pred_ada), 3))

AdaBoost accuracy: 0.792


Which model is the best and why?

In [ ]:
# Results (accuracy on the test set):
# - Bagging:            0.809
# - Pasting:            0.804
# - Random Forest:      0.815   <- best
# - Gradient Boosting:  0.807
# - AdaBoost:           0.792
# - KNN (previous lab): 0.793
#
# The best model is the Random Forest (about 81.5% correct predictions).
#
# Why?
# - It combines 100 decision trees and lets them vote. One tree alone makes mistakes,
#   but the mistakes of many different trees cancel each other out.
# - Each tree only sees random rows AND random columns, so the trees are very different
#   from each other -> the vote is more reliable than simple bagging.
# - It is also robust: it doesn't overfit as easily as a single deep tree.
#
# Almost all ensemble methods beat the KNN from the previous lab (0.793),
# so combining many models does give a better result.
# AdaBoost is the weakest here: it focuses a lot on the hard passengers,
# which can make it learn some noise.
# The differences are small (about 1-2%), so tuning the hyperparameters
# (n_estimators, max_depth, learning_rate) could still change the ranking.